# Course 5: NLP Applications
## Lecture 3: Chatbot Basics & Interactive Conversational Systems

---

### 🎯 Learning Objectives:
By the end of this lecture notebook, you will be able to:
1. Understand the 4 Chatbot Architectures: **Rule-Based, Intent & Entity-Based, Retrieval-Augmented (RAG), and Generative LLMs**.
2. Build an **Intent Classification & Entity Matching** conversational routing engine.
3. Manage **Dialogue State, Session Memory, and Multi-Turn Conversation History**.
4. Prototype interactive conversational pipelines using pre-trained dialogue models.
5. Understand the essentials of building Web UIs using **Gradio** (`gr.ChatInterface`).
6. Implement safety guardrails, prompt boundary controls, and error fallbacks.

---

### 💬 Chatbot Evolution & Taxonomies

```
1. Rule-Based / Regex      ──>  Exact pattern matching, zero flexibility (ELIZA)
2. Intent & Slot Filling   ──>  Classifies user intent + extracts entities (Rasa/Dialogflow)
3. Retrieval-Augmented     ──>  Finds relevant company knowledge base docs + synthesizes response
4. Generative / LLM Agent  ──>  Context-aware multi-turn dialogue with tool execution abilities
```


In [ ]:
# ==========================================
# Step 0: Imports & Environment Setup
# ==========================================
import warnings
warnings.filterwarnings('ignore')
import json
import re
import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

print("✅ Setup complete!")


---
## Section 1: Building an Intent & Entity-Based Support Bot

In practical customer service applications, many user questions map to known **intents** (FAQs, business policies, account management).
An Intent Matcher computes semantic similarity against known training utterances and executes the appropriate response handler.


In [ ]:
# ==========================================
# Step 1: Define Intent Knowledge Base
# ==========================================
INTENT_DATABASE = {
    "greeting": {
        "patterns": ["hello", "hi there", "good morning", "hey", "is anyone here"],
        "responses": ["Hello! How can I assist you with our NLP Application today?", "Hi there! What can I help you with?"]
    },
    "check_pricing": {
        "patterns": ["how much does it cost", "what is your price", "pricing plans", "subscription fees", "is it free"],
        "responses": ["Our Starter plan is $19/mo, and Pro is $49/mo. We also offer a 14-day free trial!"]
    },
    "request_refund": {
        "patterns": ["i want a refund", "how do i get my money back", "cancel my subscription and refund", "return payment"],
        "responses": ["You can request a refund within 30 days of purchase under Account Settings > Billing > Request Refund."]
    },
    "technical_issue": {
        "patterns": ["the app is crashing", "i cannot log in", "error 404", "bug in the dashboard", "page not loading"],
        "responses": ["I'm sorry to hear that! Please provide your account email and error screenshot so our tech team can investigate."]
    },
    "goodbye": {
        "patterns": ["bye", "goodbye", "see you later", "thanks that is all", "have a good day"],
        "responses": ["Thank you for chatting with us! Have a wonderful day!", "Goodbye! Let us know if you need anything else."]
    }
}

# Vectorize all intent patterns for semantic cosine similarity search
corpus_patterns = []
pattern_to_intent = []

for intent, data in INTENT_DATABASE.items():
    for pattern in data["patterns"]:
        corpus_patterns.append(pattern)
        pattern_to_intent.append(intent)

vectorizer = TfidfVectorizer().fit(corpus_patterns)
pattern_vectors = vectorizer.transform(corpus_patterns)
print(f"Indexed {len(corpus_patterns)} patterns across {len(INTENT_DATABASE)} intents.")


In [ ]:
# ==========================================
# Step 2: Intent Matching & Dispatcher Engine
# ==========================================
import random

def match_intent(user_message: str, threshold: float = 0.35):
    """Finds the closest intent matching the user utterance via Cosine Similarity."""
    user_vec = vectorizer.transform([user_message])
    similarities = cosine_similarity(user_vec, pattern_vectors)[0]
    
    best_match_idx = similarities.argmax()
    best_score = similarities[best_match_idx]
    
    if best_score >= threshold:
        matched_intent = pattern_to_intent[best_match_idx]
        response = random.choice(INTENT_DATABASE[matched_intent]["responses"])
        return matched_intent, best_score, response
    else:
        return "fallback", best_score, "I'm not completely sure I understood that. Could you rephrase your question or contact human support?"

# Test Intent Matcher
test_queries = [
    "Hey there, good morning!",
    "Can you tell me how much the monthly subscription is?",
    "My dashboard is broken and gives a crash error",
    "Where is the nearest spaceship landing port?" # Unseen out-of-domain query
]

print("🧪 Intent Matcher Test:")
for q in test_queries:
    intent, score, reply = match_intent(q)
    print(f"User: '{q}'")
    print(f"➡️  Intent: {intent} (Similarity: {score:.2f})")
    print(f"🤖 Bot: {reply}\n")


---
## Section 2: Multi-Turn Dialogue State Management

A single response is not enough for real conversations. Chatbots must maintain **Dialogue State** and message history (`[{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]`).


In [ ]:
# ==========================================
# Step 3: Multi-Turn Conversation Manager Class
# ==========================================
class ChatbotSession:
    def __init__(self, system_persona: str = "Support Assistant"):
        self.system_persona = system_persona
        self.history = []
        self.created_at = datetime.datetime.now()
        
    def add_user_message(self, message: str):
        self.history.append({"role": "user", "content": message, "time": str(datetime.datetime.now().strftime("%H:%M:%S"))})
        
    def add_assistant_message(self, message: str):
        self.history.append({"role": "assistant", "content": message, "time": str(datetime.datetime.now().strftime("%H:%M:%S"))})
        
    def respond(self, user_message: str) -> str:
        self.add_user_message(user_message)
        
        # Determine response via intent matcher
        intent, score, response_text = match_intent(user_message)
        
        # Personalize or add state logic if needed
        self.add_assistant_message(response_text)
        return response_text
    
    def get_formatted_transcript(self) -> str:
        output = [f"=== Chat Session ({self.system_persona}) ==="]
        for turn in self.history:
            prefix = "👤 User" if turn["role"] == "user" else "🤖 Assistant"
            output.append(f"[{turn['time']}] {prefix}: {turn['content']}")
        return "\n".join(output)

# Simulate a conversation
session = ChatbotSession(system_persona="Customer Care Bot")
session.respond("Hello!")
session.respond("I was charged twice, I want my money back please.")
session.respond("Thanks for your help, goodbye!")

print(session.get_formatted_transcript())


---
## Section 3: Generative Conversational Transformers

For open-domain dialogue, we can use generative conversational models.
Let's see how conversational models generate responses given conversation context.


In [ ]:
# ==========================================
# Step 4: Hugging Face Conversational Pipeline
# ==========================================
# We use a text-generation pipeline formatted as a conversational assistant
chat_generator = pipeline(
    "text-generation", 
    model="distilgpt2"
)

def generate_assistant_reply(conversation_history: list, new_user_input: str) -> str:
    """Formats multi-turn prompt and generates next assistant response."""
    prompt = "The following is a friendly and helpful conversation with an AI Customer Support Assistant.\n"
    for turn in conversation_history[-3:]: # Keep last 3 turns for context
        role = "User" if turn["role"] == "user" else "Assistant"
        prompt += f"{role}: {turn['content']}\n"
    prompt += f"User: {new_user_input}\nAssistant:"
    
    out = chat_generator(
        prompt, 
        max_new_tokens=40, 
        do_sample=True, 
        temperature=0.7, 
        top_p=0.9,
        pad_token_id=50256
    )[0]['generated_text']
    
    # Extract only newly generated assistant reply
    reply = out[len(prompt):].strip().split("\n")[0]
    return reply if reply else "I am here to help. Could you clarify your question?"

# Test generative assistant
history_context = [
    {"role": "user", "content": "Hi, what does your company do?"},
    {"role": "assistant", "content": "We build cutting-edge natural language processing tools for businesses."}
]
new_input = "Can you help me summarize customer feedback?"
gen_reply = generate_assistant_reply(history_context, new_input)
print(f"👤 User: {new_input}")
print(f"🤖 Generative Assistant: {gen_reply}")


---
## Section 4: Web UI Prototyping Basics (Gradio Overview)

Why **Gradio** is the standard for NLP application prototypes:
- **Zero HTML/CSS/JavaScript needed**: Pure Python web components.
- **Built-in `gr.ChatInterface`**: Out-of-the-box streaming chat UI.
- **Instant Shareable Links**: Test with stakeholders anywhere via public tunneling.

Here is what the basic Gradio chat structure looks like:

```python
import gradio as gr

def echo_chatbot(message, history):
    intent, score, reply = match_intent(message)
    return reply

demo = gr.ChatInterface(
    fn=echo_chatbot,
    title="Customer Care AI Assistant",
    description="Ask questions regarding subscriptions, technical issues, or refunds.",
    examples=["How much is the subscription?", "I need a refund", "App is crashing"]
)
# demo.launch()
```

In the upcoming lab and deployment scripts, we will launch a complete multi-tab application covering Classification, Summarization, and Chatbots! 🚀
